# Bidirectional Longitudinal Associations Between Intra-Team Peer Ostracism and Intra-Team Loneliness Among Adolescent Athletes: Evidence from Follow-Up and Training-Diary Data Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.jfpk-yvfm/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Dataset description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore available record sets
record_sets = metadata.recordSet

if not record_sets:
    print("No record sets are defined in this dataset metadata.")
else:
    print("Record Sets (@id):")
    for rs in record_sets:
        print(f"  - {rs['@id']}")

# Review fields and columns by @id for each record set
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    fields = rs.get('field', [])
    print("  Fields (@id):")
    for field in fields:
        print(f"    - {field['@id']}")
    columns = rs.get('column', [])
    print("  Columns (@id):")
    for column in columns:
        print(f"    - {column['@id']}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract data from each record set
dataframes = {}
record_sets_ids = []

# Collect record set @id's for extraction
if record_sets:
    for rs in record_sets:
        record_sets_ids.append(rs['@id'])

for record_set_id in record_sets_ids:
    # Load records using mlcroissant with record set @id
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nDataFrame for record set {record_set_id}: Columns=")
        print(df.columns.tolist())
        print(df.head())
    else:
        print(f"No records found for record set {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# For demonstration, pick the first available DataFrame
if dataframes:
    # Use the first record set id and DataFrame
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]

    # List available columns
    print("Available columns:", df.columns.tolist())

    # Select a numeric field for EDA (if available)
    numeric_field = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break
    
    if numeric_field:
        threshold = df[numeric_field].mean()
        # Filter records above mean
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt grouping by a categorical column
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].nunique() > 1 and not np.issubdtype(df[col].dtype, np.number):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical group field found for grouping.")
    else:
        print("No numeric fields are available for EDA.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field found, plot group-wise means
    if group_field:
        plt.figure(figsize=(8,4))
        sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field])
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to load and explore the dataset using the `mlcroissant` library. By referencing and extracting entities via their `@id`, we ensured reproducibility and accurate mapping of metadata to records. Basic exploratory analysis and visualization steps highlighted patterns and distributions within the available data. For a deeper study, consult the dataset documentation and expand analyses to additional record sets and fields as required.